**Simulación de un Transformer (Mecanismo de Autoatención)**



Paso 1: Word Embeddings Iniciales

In [ ]:
import numpy as np

# --- CONFIGURACIÓN Y EMBEDDINGS ---
# Creamos los vectores base (embeddings) que representan cada palabra en un espacio de 4 dimensiones.

word_embeddings = {
    'no':        np.array([0.2, 0.3, 0.1, 0.4]),  # negación, leve intensidad
    'temas':     np.array([0.1, 0.7, 0.2, 0.6]),  # relacionado con miedo
    'ir':        np.array([0.8, 0.1, 0.0, 0.5]),  # movimiento
    'despacio':  np.array([0.6, 0.1, 0.1, 0.4]),  # movimiento lento
    'solo':      np.array([0.2, 0.1, 0.1, 0.3]),  # palabra funcional
    'ten':       np.array([0.3, 0.1, 0.1, 0.3]),  # verbo auxiliar
    'miedo':     np.array([0.1, 0.9, 0.2, 0.8]),  # emoción fuerte
    'de':        np.array([0.1, 0.1, 0.1, 0.2]),  # palabra funcional
    'quedarte':  np.array([0.2, 0.2, 0.6, 0.5]),  # relacionado con "no avanzar"
    'quieto':    np.array([0.0, 0.2, 0.9, 0.7])   # inmovilidad total
}


Step 2: Creamos la matriz de entrada X

In [ ]:
# Apilamos los vectores en una matriz donde cada fila es una palabra de la secuencia.
X = np.vstack([
    word_embeddings['no'],
    word_embeddings['temas'],
    word_embeddings['ir'],
    word_embeddings['despacio'],
    word_embeddings['solo'],
    word_embeddings['ten'],
    word_embeddings['miedo'],
    word_embeddings['de'],
    word_embeddings['quedarte'],
    word_embeddings['quieto']
])


Step 3: Definimos matrices de pesos W_q, W_k, and W_v

In [ ]:
# Matrices de pesos que transformarán el input en Query (qué busco), Key (qué ofrezco) y Value (contenido).

W_q = np.array([
    [0.9, 0.1, 0.0, 0.0],  # dimensión 1 influenciada por un poco de emoción
    [0.1, 0.9, 0.1, 0.0],  # emoción mezclada con inmovilidad
    [0.0, 0.1, 0.9, 0.1],  # inmovilidad influida ligeramente por intensidad
    [0.0, 0.0, 0.1, 0.9]   # intensidad casi intacta
])

W_k = np.array([
    [0.9, 0.0, 0.1, 0.0],  # movimiento correlacionado con inmovilidad
    [0.0, 0.9, 0.1, 0.0],  # emoción influye ligeramente en la inmovilidad
    [0.1, 0.1, 0.9, 0.1],  # inmovilidad recibe señales de emoción y movimiento
    [0.0, 0.0, 0.1, 0.9]   # intensidad influye ligeramente en inmovilidad
])

W_v = np.array([
    [0.7, 0.1, 0.0, 0.1],  # movimiento aporta pero no domina
    [0.2, 0.8, 0.2, 0.1],  # emoción tiene peso fuerte en los valores
    [0.0, 0.1, 0.8, 0.2],  # inmovilidad tiene impacto importante
    [0.1, 0.0, 0.0, 0.6]   # intensidad se mantiene
])


Step 4: Calcular las matrices Q, K y V

In [ ]:
# Proyectamos el input X hacia los espacios de Query, Key y Value.
Q = np.dot(X, W_q)
K = np.dot(X, W_k)
V = np.dot(X, W_v)

Step 5: Calcular Raw Attention Scores

In [ ]:
# Calculamos la similitud (producto punto) entre cada Query y cada Key.
scores = np.dot(Q, K.T)

Step 6: Escalar puntajes

In [ ]:
# Dividimos por la raíz de la dimensión de las Keys para estabilizar el gradiente.
d_k = K.shape[1]
scaled_scores = scores / np.sqrt(d_k)

Step 7: Aplicar softmax para obtener los pesos de atención

In [ ]:
# Convertimos los scores en probabilidades que suman 1 por cada fila.
exp_scores = np.exp(scaled_scores)
attention_weights = exp_scores / exp_scores.sum(axis=1, keepdims=True)

In [ ]:
print(attention_weights)

[[0.09663386 0.1060997  0.10045192 0.09783958 0.09228478 0.09323556
  0.11305554 0.08976357 0.10272744 0.10790802]
 [0.09450217 0.11434529 0.0941155  0.09193877 0.08643328 0.08719289
  0.12840929 0.08341872 0.10441494 0.11522915]
 [0.095221   0.10002304 0.11561903 0.10700767 0.09181269 0.09490729
  0.10588997 0.08701239 0.10109398 0.10141293]
 [0.09566495 0.10082657 0.11032184 0.10425626 0.09254564 0.09493116
  0.10604907 0.08866826 0.10239132 0.10434492]
 [0.09734682 0.1023686  0.10208747 0.09978122 0.09491009 0.09578729
  0.10639728 0.0928031  0.10246077 0.10605735]
 [0.09704393 0.10187165 0.10412725 0.10100436 0.09452967 0.09579052
  0.10597611 0.09205753 0.10235158 0.1052474 ]
 [0.09300025 0.1185817  0.09212093 0.08938046 0.08296776 0.08378903
  0.1376621  0.07932491 0.10467414 0.11849872]
 [0.09808407 0.1023732  0.10019066 0.0990005  0.09611784 0.09660929
  0.10542765 0.09475787 0.10214823 0.10529068]
 [0.09274125 0.10574848 0.09618885 0.09449183 0.08773434 0.08879794
  0.11472792

Step 8: Calcular salida final

In [ ]:
# Multiplicamos los pesos de atención por los Valores (V) para obtener el contexto ponderado.
output = np.dot(attention_weights, V)

In [ ]:
print("Final output matrix (10 x 4):")
print(output)

Final output matrix (10 x 4):
[[0.28654163 0.28617741 0.25854729 0.39456391]
 [0.28431137 0.30031513 0.26984365 0.4044659 ]
 [0.29510533 0.27849624 0.24936952 0.3907812 ]
 [0.29177195 0.27924269 0.25248995 0.39128316]
 [0.2867022  0.28010007 0.25476732 0.39057695]
 [0.28796085 0.27957434 0.25381112 0.39042413]
 [0.28408909 0.30832636 0.27527787 0.41002576]
 [0.28549266 0.27931432 0.25407949 0.38937787]
 [0.2827624  0.28946944 0.27341691 0.40247202]
 [0.27769922 0.29557308 0.28807095 0.4106366 ]]


Step 9: Visualización de los pesos de atención

In [ ]:
tokens = ["no", "temas", "ir", "despacio", "solo",
          "ten", "miedo", "de", "quedarte", "quieto"]

In [ ]:
import numpy as np

def visualize_attention(attention_weights, tokens):
    print("Attention Weight Visualization:\n")
    n = len(tokens)
    for i in range(n):
        print(f"{tokens[i]} attending to:")
        for j in range(n):
            w = attention_weights[i, j]
            bar = "█" * int(w * 40)   # escala las barras (40 se puede ajustar)
            print(f"  {tokens[j]:10s} {bar:<40s} ({w:.3f})")
        print("\n" + "-" * 60 + "\n")


In [ ]:
visualize_attention(attention_weights, tokens)

Attention Weight Visualization:

no attending to:
  no         ███                                      (0.097)
  temas      ████                                     (0.106)
  ir         ████                                     (0.100)
  despacio   ███                                      (0.098)
  solo       ███                                      (0.092)
  ten        ███                                      (0.093)
  miedo      ████                                     (0.113)
  de         ███                                      (0.090)
  quedarte   ████                                     (0.103)
  quieto     ████                                     (0.108)

------------------------------------------------------------

temas attending to:
  no         ███                                      (0.095)
  temas      ████                                     (0.114)
  ir         ███                                      (0.094)
  despacio   ███                                      (0.092)

**Cuestionario**

1. En la arquitectura transformer, ¿qué representa Nx y cómo se utiliza?

En el Transformer, Nx indica que el bloque estructural (Encoder Layer en el encoder o Decoder Layer en el decoder) se repite N veces. En el modelo base, ambos encoder y decoder repiten su respectivo bloque 6 veces.

Se utiliza para crear la profundidad del modelo: cada capa toma la salida de la anterior y la transforma, permitiendo al Transformer aprender representaciones cada vez más complejas.